# Kalman Filter with Time-Point Priors and Positive Coefficients

In [ ]:
import pandas as pd
import numpy as np

import numpyro
numpyro.set_host_device_count(4)
import jax.numpy as jnp

from numpyro import distributions as dist
from numpyro.infer import MCMC, NUTS

import matplotlib.pyplot as plt
from jax import random

from wunkui.models.ssp import kalman_filter_1d

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
USE_DISCLOSURE = False # False → plain Kalman filter; True → enforce a_obs at sampled windows
N_PERIODS      = 3     # number of disclosure windows to sample
N_POINTS       = 10      # consecutive steps per window

In [ ]:
df = pd.read_csv(
    '../resource/simple/df.csv', parse_dates=['date']
)
# for quick testing
df = df[-365:].reset_index(drop=True)

saturation_df = pd.read_csv(
    '../resource/simple/saturation_df.csv',
)
coefs_df = pd.read_csv(
    '../resource/simple/coefs_df.csv',
)

regressors = saturation_df["regressor"].tolist()
response = df["sales"].values

response_norm = response.mean()
y = np.log(response / response_norm)
y = jnp.array(y)
X = df[regressors].values
sat_arr = saturation_df.set_index("regressor").loc[regressors, "saturation"].values
X = np.log1p(X / sat_arr)

Z = jnp.concatenate([jnp.ones((X.shape[0], 1)), jnp.array(X)], -1)

positivity_idx = jnp.array([0] + [1] * len(regressors))

print("y shape", y.shape)
print("Z shape", Z.shape)
print("X shape", X.shape)

In [ ]:
# df = pd.read_csv(
#     '../resource/sparse/sparse_df.csv', parse_dates=['date']
# )
# regressors = [
#     "ch_a1",
#     "ch_b1",
#     "ch_b2",
#     "ch_a2",
#     "ch_b3",
#     "ch_b4",
#     "ch_a3",
#     "ch_a4",
#     "ch_b5",
#     "ch_b6",
#     "ch_b7",
#     "ch_b8",
#     "ch_a5",
# ]
# coefs_df = None

# df = df.groupby("date")[["outcome"] + regressors].sum().reset_index()

# response = df["outcome"].values
# response_mean = response.mean()
# response_std = response.std()

# y = (response - response_mean) / response_std
# y = jnp.array(y)

# X = df[regressors].values
# X = (X - X.mean(axis=0)) / X.std(axis=0)
# Z = jnp.concatenate([jnp.ones((X.shape[0], 1)), jnp.array(X)], -1)

# positivity_idx = jnp.array([0] + [1] * len(regressors))

# print("y shape", y.shape)
# print("Z shape", Z.shape)
# print("X shape", X.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df["date"], y, label="y")

In [ ]:
def make_fourier_series(t, period, order):
    """
    Args
    ----
    t: array-like, time points at which to evaluate the Fourier series
    period: int, the period of the seasonality
    order: int, the number of Fourier terms to include
    """
    sin_terms = jnp.array([jnp.sin(2 * jnp.pi * i * t / period) for i in range(1, order + 1)])
    cos_terms = jnp.array([jnp.cos(2 * jnp.pi * i * t / period) for i in range(1, order + 1)])
    return jnp.concatenate((sin_terms, cos_terms), axis=0).transpose(1, 0)

x_glb_trend = jnp.arange(len(y)) / 365.25
x_annual_seas = make_fourier_series(jnp.arange(len(y)), period=365.25, order=3)
x_weekly_seas = make_fourier_series(jnp.arange(len(y)), period=7, order=2)
x_seas = jnp.concatenate((x_annual_seas, x_weekly_seas), axis=1)
print(x_seas.shape)

In [ ]:
def disclose_at(
    n_steps: int,
    n_states: int,
    true_state: jnp.ndarray,
    regressors: list,
    n_periods: int = 3,
    n_points: int = 7,
    seed: int = 42,
    obs_scale: float = 0.001,
) -> tuple:
    """Construct a_obs_loc and a_obs_var by disclosing the ground-truth latent
    state over n_periods randomly drawn windows of n_points consecutive steps.

    Parameters
    ----------
    n_steps : int
        Total number of time steps.
    n_states : int
        Number of latent states (intercept + regressors).
    true_state : jnp.ndarray, shape (n_states,)
        Ground-truth state vector; intercept entry is ignored (var stays inf).
    regressors : list[str]
        Regressor names; determines which state indices get a finite variance.
    n_periods : int
        Number of disclosure windows to draw.
    n_points : int
        Number of consecutive steps per window.
    seed : int
        RNG seed for reproducibility.
    obs_scale : float
        Standard deviation expressing confidence in the disclosed state.
        Smaller → tighter prior; larger → more diffuse.

    Returns
    -------
    a_obs_loc : jnp.ndarray, shape (n_steps, n_states)
        Disclosed state means; zero where not disclosed.
    a_obs_var : jnp.ndarray, shape (n_steps, n_states)
        Disclosed state variances; inf where no information is provided
        (intercept column always inf, undisclosed timesteps always inf).
    obs_idx : np.ndarray
        Sorted array of all disclosed time indices (for plotting).
    """
    rng = np.random.default_rng(seed)
    # sample n_periods window starts; ensure each window fits within n_steps
    starts = rng.choice(n_steps - n_points + 1, size=n_periods, replace=False)
    obs_idx = np.unique(np.concatenate([np.arange(s, s + n_points) for s in starts]))

    a_obs_loc = jnp.zeros((n_steps, n_states))
    # default inf = zero precision = no information; pure filter carries through
    a_obs_var = jnp.full((n_steps, n_states), jnp.inf)

    a_obs_loc = a_obs_loc.at[obs_idx].set(true_state)
    # intercept has no ground truth so its variance stays inf
    var_row = jnp.array([jnp.inf] + [obs_scale ** 2] * len(regressors))
    a_obs_var = a_obs_var.at[obs_idx].set(var_row)

    return a_obs_loc, a_obs_var, obs_idx

In [ ]:
n_steps = y.shape[0]
n_states = Z.shape[-1]

if coefs_df is not None:
    # ground-truth state from coefs_df; intercept entry left as 0 (unknown)
    coefs_lookup = coefs_df.set_index("regressor")["coef"]
    true_state = jnp.array([0.0] + [coefs_lookup[r] for r in regressors])

    if USE_DISCLOSURE:
        a_obs_loc, a_obs_var, obs_idx = disclose_at(
            n_steps=n_steps,
            n_states=n_states,
            true_state=true_state,
            regressors=regressors,
            n_periods=N_PERIODS,
            n_points=N_POINTS,
        )
        print("n disclosed steps:", len(obs_idx))
    else:
        a_obs_loc = a_obs_var = None
        obs_idx = np.array([], dtype=int)
        print("disclosure disabled")

    print("a_obs_loc:", a_obs_loc if a_obs_loc is None else a_obs_loc.shape)
    print("a_obs_var:", a_obs_var if a_obs_var is None else a_obs_var.shape)
else:
    a_obs_loc = a_obs_var = None
    obs_idx = np.array([], dtype=int)
    print("no ground truth state available; disclosure disabled")

In [ ]:
a0 = jnp.array([0.0] + [0.1] * (X.shape[-1]))
P0 = jnp.array([1.0] + [0.1] * (X.shape[-1]))

# 2-element priors: [level, shared_regressors]
sigma_q_loc_prior = np.array([0.05, 0.01])
sigma_q_scale_prior = np.array([0.05, 0.01])

print("a0 shape:", a0.shape)
print("P0 shape:", P0.shape)

In [ ]:
# Test run to verify dimensions are correct before sampling
sigma_h = jnp.array(0.1)
sigma_q_raw = jnp.array([0.01, 0.01])  # [level, shared_regressors]
sigma_q = jnp.concatenate([sigma_q_raw[:1], jnp.repeat(sigma_q_raw[1:], Z.shape[-1] - 1)])
print("sigma_q expanded:", sigma_q.shape, sigma_q)

lp, at, Pt, _, _, _ = kalman_filter_1d(
    a0=a0,
    P0=P0,
    sigma_h=sigma_h,
    sigma_q=sigma_q,
    y=y,
    Z=Z,
    logp=True,
    a_obs_loc=a_obs_loc,
    a_obs_var=a_obs_var,
    positivity_idx=positivity_idx,
)
print("lp:", lp)
print("at shape:", at.shape)
print("Pt shape:", Pt.shape)

In [ ]:
def _nuts_fn(a0, P0):
    sigma_h = numpyro.sample(
        'sigma_h',
        dist.TruncatedNormal(
            0.1, 1.0, high=1.0, low=1e-5
        )
    )

    # sample 2-element sigma_q: [level, shared_regressors]
    sigma_q_raw = numpyro.sample(
        'sigma_q',
        dist.TruncatedNormal(
            sigma_q_loc_prior, 
            sigma_q_scale_prior, 
            high=0.1, 
            low=1e-5
        )
    )

    # expand to full n_states: [level, reg1, reg2, ..., regN]
    n_regressors = Z.shape[-1] - 1
    sigma_q = jnp.concatenate([sigma_q_raw[:1], jnp.repeat(sigma_q_raw[1:], n_regressors)])

    lp, at, _, _, _, _ = kalman_filter_1d(
        a0=a0,
        P0=P0,
        sigma_h=sigma_h,
        sigma_q=sigma_q,
        y=y,
        Z=Z,
        logp=True,
        a_obs_loc=a_obs_loc,
        a_obs_var=a_obs_var,
        positivity_idx=positivity_idx,
    )

    numpyro.factor("lp", lp)
    numpyro.deterministic("at", at)
    numpyro.deterministic("mu", jnp.sum(Z * at, -1))

In [ ]:
nuts_kernel = NUTS(_nuts_fn)
# kernel = HMCGibbs(
#     inner_kernel=nuts_kernel,
#     gibbs_fn=_gibbs_fn,
#     gibbs_sites=["smooth_alpha"],
# )
mcmc = MCMC(
    nuts_kernel, 
    num_warmup=400, 
    num_samples=400,
    num_chains=4,
)
rng_key = random.PRNGKey(0)
rng_keys = random.split(rng_key, 2)
rng_key = rng_keys[0]
mcmc_rng_key = rng_keys[1]
mcmc.run(mcmc_rng_key, a0, P0)

In [ ]:
from numpyro.diagnostics import summary

# print_summary covers R-hat and n_eff for all sites
mcmc.print_summary()

# detailed per-parameter diagnostics for scalar params
samples = mcmc.get_samples(group_by_chain=True)
diag = summary(samples, prob=0.9)

# flag any R-hat > 1.01 or n_eff < 100
# use max(r_hat) / min(n_eff) for multi-dimensional params (at, lam, mu, etc.)
print("\n--- Convergence flags ---")
for param, stats in diag.items():
    rhat = float(np.asarray(stats["r_hat"]).max())
    neff = float(np.asarray(stats["n_eff"]).min())
    if rhat > 1.01 or neff < 100:
        print(f"  WARNING  {param:<20}  r_hat={rhat:.4f}  n_eff={neff:.1f}")

In [ ]:
posteriors_dict = mcmc.get_samples()
posteriors_dict.keys()

In [ ]:
state_labels = ["intercept"] + regressors
coefs_lookup = coefs_df.set_index("regressor")["coef"] if coefs_df is not None else None

obs_dates = df["date"].values[obs_idx] if len(obs_idx) > 0 else []

# posterior quantiles of at across MCMC samples: each (n_steps, n_states)
at_lo, at_mid, at_hi = np.quantile(posteriors_dict["at"], q=[0.05, 0.5, 0.95], axis=0)

fig, axes = plt.subplots(len(state_labels), 1, figsize=(12, 2.5 * len(state_labels)), sharex=True)
for i, (ax, label) in enumerate(zip(axes, state_labels)):
    ax.plot(df["date"], at_mid[:, i], linewidth=0.8)
    ax.fill_between(df["date"], at_lo[:, i], at_hi[:, i], alpha=0.25, label="90% CI")
    if i > 0 and coefs_lookup is not None and label in coefs_lookup.index:
        ax.axhline(coefs_lookup[label], color="red", linestyle="--", linewidth=1.0, label="true coef")
    # mark time points where this state was disclosed (finite variance = active disclosure)
    if USE_DISCLOSURE and len(obs_idx) > 0 and np.any(np.isfinite(np.array(a_obs_var)[obs_idx, i])):
        first = True
        for d in obs_dates:
            ax.axvline(d, color="orange", linestyle=":", linewidth=0.9, alpha=0.8,
                       label="truth disclosed" if first else None)
            first = False
    ax.legend(fontsize=8)
    ax.set_title(label, fontsize=9, loc="left")
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel("date")
title = "Filtered state coefficients — disclosure ON" if USE_DISCLOSURE else "Filtered state coefficients — disclosure OFF"
fig.suptitle(title, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
mu_samples = np.array(posteriors_dict["mu"])
eps_samples = np.random.normal(
    0,
    np.array(posteriors_dict["sigma_h"])[:, None],
    size=mu_samples.shape,
)

# compute p5, p50, p95 for forecast
yhat_lower, yhat_mid, yhat_upper = np.quantile(np.exp(mu_samples + eps_samples) * response_norm, q=[0.05, 0.5, 0.95], axis=0)
# yhat_lower, yhat_mid, yhat_upper = np.quantile((mu_samples + eps_samples) * response_std + response_mean, q=[0.05, 0.5, 0.95], axis=0)

In [ ]:
import matplotlib.pyplot as plt
n = 100
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(np.arange(n), response[-n:], label='Observed', alpha=0.5, color="orange", s=10)
ax.plot(np.arange(n), yhat_mid[-n:], label='Forecast', linestyle='--', alpha=0.8, color="dodgerblue")
ax.fill_between(np.arange(n), yhat_lower[-n:], yhat_upper[-n:], alpha=0.5, label='95% Prediction Interval', color="dodgerblue")
ax.set_title('State Space Model Forecast')